In [ ]:
import json
import os
import shlex
import subprocess
import time
from pathlib import Path
from openai import OpenAI

ROOT = Path(r"C:\Users\rrcar\Documents\drIpTECH")
BRIDGE = ROOT / ".copilot_bridge"
REQ = BRIDGE / "request.json"
RES = BRIDGE / "response.json"
BRIDGE.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
model = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")
venv_py = ROOT / ".venv" / "Scripts" / "python.exe"

history = [{"role": "system", "content": "Notebook bridge assistant"}]

def run_pwsh(cmd: str) -> str:
    p = subprocess.run(
        ["pwsh", "-NoProfile", "-Command", cmd],
        cwd=str(ROOT),
        capture_output=True,
        text=True,
        timeout=180
    )
    out = (p.stdout or "").strip()
    err = (p.stderr or "").strip()
    if p.returncode != 0:
        return f"[pwsh exit {p.returncode}]\n{err or out or '(no output)'}"
    return out or "(no output)"

def run_py(args: str) -> str:
    p = subprocess.run(
        [str(venv_py)] + shlex.split(args, posix=False),
        cwd=str(ROOT),
        capture_output=True,
        text=True,
        timeout=180
    )
    out = (p.stdout or "").strip()
    err = (p.stderr or "").strip()
    if p.returncode != 0:
        return f"[python exit {p.returncode}]\n{err or out or '(no output)'}"
    return out or "(no output)"

def run_chat(text: str) -> str:
    history.append({"role": "user", "content": text})
    r = client.chat.completions.create(model=model, messages=history, timeout=60)
    out = r.choices[0].message.content or ""
    history.append({"role": "assistant", "content": out})
    return out

last_mtime = 0.0
print(f"Bridge worker running. Request file: {REQ}")
while True:
    try:
        if REQ.exists() and REQ.stat().st_mtime > last_mtime:
            last_mtime = REQ.stat().st_mtime
            data = json.loads(REQ.read_text(encoding="utf-8"))
            kind = (data.get("kind") or "chat").strip().lower()
            text = (data.get("text") or "").strip()

            if kind == "pwsh":
                output = run_pwsh(text)
            elif kind == "py":
                output = run_py(text)
            else:
                output = run_chat(text)

            RES.write_text(
                json.dumps({"ok": True, "kind": kind, "output": output}, ensure_ascii=False, indent=2),
                encoding="utf-8"
            )
    except Exception as e:
        RES.write_text(
            json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
    time.sleep(0.35)

Bridge worker running. Request file: C:\Users\rrcar\Documents\drIpTECH\.copilot_bridge\request.json
